# Distributed Training Infrastructure

Why distributed: model won't fit on one GPU (model parallelism), or training on one GPU is too slow (data parallelism), or data preprocessing is bottlenecked (distributed data).

Cover: DDP, FSDP, Ray Data, orchestration overview (Airflow/Prefect), K8s for ML.

VRAM: mostly conceptual, ~2 GB for demos. Time: ~2-3 hours.

In [ ]:
import os
import time
import math
import multiprocessing
from dataclasses import dataclass, field
from typing import Callable, Any
from pathlib import Path

import torch
import torch.nn as nn
import torch.distributed  # conceptual — won't init without multi-GPU env
import matplotlib.pyplot as plt
import numpy as np

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Parallelism (DDP)

`DistributedDataParallel` is the standard approach when the model fits on a single GPU but training on one GPU is too slow.

**Mechanics:**
- Each GPU gets a full model replica + a shard of the data batch
- Forward pass: each GPU processes its shard independently
- Backward pass: gradients are all-reduced (averaged) across GPUs before the optimizer step
- Key concepts: `process group`, `rank` (this GPU's index), `world_size` (total GPU count), `DistributedSampler` (ensures non-overlapping data shards)

**Communication pattern: ring all-reduce**
- O(2N) total bytes transferred, independent of number of GPUs
- Each GPU only communicates with its two neighbors in the ring — doesn't bottleneck as GPUs are added

**DDP vs `DataParallel` (old API):**
| | DDP | DataParallel |
|---|---|---|
| Process model | Multi-process (1 process per GPU) | Multi-thread (1 process) |
| GIL bottleneck | No | Yes |
| Recommended | Yes | No |

**Launch:** `torchrun --nproc_per_node=4 train.py` — spawns 4 processes, each sees one GPU.

In [ ]:
# ---------------------------------------------------------------------------
# DDP boilerplate — this is the full pattern you'd write in a real training
# script. Won't actually distribute (no multi-GPU env here), but shows the
# complete structure.
# ---------------------------------------------------------------------------

def setup(rank: int, world_size: int, backend: str = "nccl") -> None:
    """Initialize the distributed process group."""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    # torch.distributed.init_process_group(backend, rank=rank, world_size=world_size)
    print(f"[rank {rank}] process group initialized (backend={backend})")


def cleanup() -> None:
    """Tear down the process group."""
    # torch.distributed.destroy_process_group()
    print("process group destroyed")


class ToyModel(nn.Module):
    """Minimal model for DDP illustration."""
    def __init__(self, in_features: int = 128, out_features: int = 10) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, out_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def ddp_training_step_pseudocode(rank: int, world_size: int) -> None:
    """Full DDP training pattern for a single process (rank)."""
    setup(rank, world_size)

    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")
    model = ToyModel().to(device)

    # Wrap with DDP — all-reduce happens automatically in .backward()
    # model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[rank])

    # DistributedSampler partitions indices so each rank sees disjoint data
    # dataset = MyDataset()
    # sampler = torch.utils.data.DistributedSampler(
    #     dataset, num_replicas=world_size, rank=rank, shuffle=True
    # )
    # loader = DataLoader(dataset, batch_size=32, sampler=sampler)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Simulated training loop
    for epoch in range(3):
        # sampler.set_epoch(epoch)  # ensures different shuffle each epoch
        x = torch.randn(32, 128, device=device)
        y = torch.randint(0, 10, (32,), device=device)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()  # with DDP: triggers all-reduce of gradients here
        optimizer.step()

        if rank == 0:  # only rank 0 logs/checkpoints
            print(f"epoch {epoch} | loss {loss.item():.4f}")

    cleanup()


# Run single-GPU simulation (rank=0, world_size=1)
ddp_training_step_pseudocode(rank=0, world_size=1)


# ---------------------------------------------------------------------------
# Theoretical throughput scaling with DDP
# Near-linear up to the point where communication overhead dominates
# ---------------------------------------------------------------------------

@dataclass
class DDPScalingConfig:
    param_count: int          # number of model parameters
    bytes_per_param: int = 4  # float32
    bandwidth_gbps: float = 600.0   # NVLink bandwidth (A100 peer)
    compute_time_per_step_ms: float = 100.0  # single-GPU baseline


def simulate_ddp_scaling(
    cfg: DDPScalingConfig,
    gpu_counts: list[int],
) -> dict[str, list[float]]:
    """Estimate throughput scaling for DDP across different GPU counts.

    Ring all-reduce transfers 2*(N-1)/N * gradient_bytes per GPU.
    Communication time = bytes / bandwidth.
    Effective step time = max(compute, communication) (assuming overlap).
    Throughput = samples_per_step / step_time.
    """
    gradient_bytes = cfg.param_count * cfg.bytes_per_param
    bandwidth_bytes_per_s = cfg.bandwidth_gbps * 1e9 / 8

    throughputs: list[float] = []
    efficiencies: list[float] = []

    baseline_samples_per_step = 32  # per GPU

    for n in gpu_counts:
        # Ring all-reduce: each GPU sends/receives 2*(N-1)/N * total_bytes
        allreduce_bytes = gradient_bytes * 2 * (n - 1) / n if n > 1 else 0
        comm_time_ms = (allreduce_bytes / bandwidth_bytes_per_s) * 1000

        # PyTorch overlaps backward + all-reduce, so effective overhead is partial
        overlap_factor = 0.3  # ~30% of comm overlaps with compute
        effective_comm_ms = comm_time_ms * (1 - overlap_factor)

        step_time_ms = cfg.compute_time_per_step_ms + effective_comm_ms
        total_samples = baseline_samples_per_step * n
        throughput = total_samples / (step_time_ms / 1000)  # samples/sec

        ideal_throughput = (baseline_samples_per_step / (cfg.compute_time_per_step_ms / 1000)) * n
        efficiency = throughput / ideal_throughput * 100

        throughputs.append(throughput)
        efficiencies.append(efficiency)

    return {"gpu_counts": gpu_counts, "throughputs": throughputs, "efficiencies": efficiencies}


# 100M param model, 600 Gbps NVLink
cfg_small = DDPScalingConfig(param_count=100_000_000, compute_time_per_step_ms=50)
# 7B param model — more communication overhead
cfg_large = DDPScalingConfig(param_count=7_000_000_000, compute_time_per_step_ms=500)

gpu_counts = [1, 2, 4, 8, 16, 32, 64]
results_small = simulate_ddp_scaling(cfg_small, gpu_counts)
results_large = simulate_ddp_scaling(cfg_large, gpu_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ideal = [results_small["throughputs"][0] * n for n in gpu_counts]
ax1.plot(gpu_counts, ideal, "k--", label="ideal (linear)", alpha=0.5)
ax1.plot(gpu_counts, results_small["throughputs"], "b-o", label="100M params")
ax1.plot(gpu_counts, results_large["throughputs"], "r-s", label="7B params")
ax1.set_xlabel("GPU count")
ax1.set_ylabel("Throughput (samples/sec)")
ax1.set_title("DDP Throughput vs GPU Count")
ax1.legend()
ax1.set_xscale("log", base=2)
ax1.set_xticks(gpu_counts)
ax1.set_xticklabels(gpu_counts)

ax2.plot(gpu_counts, results_small["efficiencies"], "b-o", label="100M params")
ax2.plot(gpu_counts, results_large["efficiencies"], "r-s", label="7B params")
ax2.axhline(80, color="gray", linestyle="--", alpha=0.5, label="80% efficiency threshold")
ax2.set_xlabel("GPU count")
ax2.set_ylabel("Scaling efficiency (%)")
ax2.set_title("DDP Scaling Efficiency")
ax2.legend()
ax2.set_xscale("log", base=2)
ax2.set_xticks(gpu_counts)
ax2.set_xticklabels(gpu_counts)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()
print("Small model scales well; large model hits comm bottleneck earlier.")

## Fully Sharded Data Parallelism (FSDP)

When the model itself doesn't fit on one GPU, DDP fails at the replica-creation step. FSDP solves this.

**What FSDP shards:**
- Model parameters: each GPU holds only 1/N of each layer's parameters
- Gradients: sharded, not replicated
- Optimizer states: sharded (this is the big win — Adam states are 2x param count in float32)

**Forward pass mechanics:**
1. Before each layer: all-gather parameters from all GPUs (temporary full layer in memory)
2. Run forward through layer
3. Discard non-owned parameter shards (memory freed immediately)

**Backward pass mechanics:**
1. All-gather parameters again for each layer
2. Compute gradients
3. Reduce-scatter gradients (each GPU keeps only its shard)
4. Discard non-owned parameter shards

**Sharding strategies:**
| Strategy | Params | Grads | Optimizer | Memory savings |
|---|---|---|---|---|
| `FULL_SHARD` | sharded | sharded | sharded | Maximum |
| `SHARD_GRAD_OP` | replicated after fwd | sharded | sharded | Moderate |
| `NO_SHARD` | replicated | replicated | replicated | None (= DDP) |

**Memory example — 7B param model with Adam on 8x A100-80GB:**
- No sharding: ~120 GB per GPU (params + grads + optimizer states) — doesn't fit
- FSDP `FULL_SHARD`: ~15 GB per GPU — fits with room to spare

**`auto_wrap_policy`:** Tells FSDP which submodules to wrap independently. Common: wrap by parameter count threshold, or wrap each Transformer block as its own FSDP unit.

In [ ]:
# ---------------------------------------------------------------------------
# FSDP wrapping pattern (conceptual — requires multi-GPU to actually run)
# ---------------------------------------------------------------------------

# from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
# from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
# from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
# import functools
#
# model = LargeTransformer()  # doesn't fit on one GPU
#
# # Wrap individual transformer blocks (each block becomes an FSDP unit)
# auto_wrap = functools.partial(
#     size_based_auto_wrap_policy,
#     min_num_params=1e8,  # wrap any module with >=100M params
# )
#
# model = FSDP(
#     model,
#     sharding_strategy=ShardingStrategy.FULL_SHARD,
#     auto_wrap_policy=auto_wrap,
#     mixed_precision=MixedPrecision(param_dtype=torch.bfloat16),
#     device_id=torch.cuda.current_device(),
# )
#
# # Training loop is identical to DDP from here
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
# for batch in dataloader:
#     loss = model(batch)
#     loss.backward()
#     optimizer.step()
#     optimizer.zero_grad()

print("FSDP wrapping pattern shown above (commented).")
print("Key: auto_wrap_policy determines FSDP unit granularity.")


# ---------------------------------------------------------------------------
# Memory estimator
# ---------------------------------------------------------------------------

@dataclass
class MemoryEstimate:
    params_gb: float
    gradients_gb: float
    optimizer_states_gb: float
    activations_gb: float
    total_gb: float


def estimate_memory_per_gpu(
    param_count: int,
    optimizer: str,
    sharding_strategy: str,
    num_gpus: int,
    param_dtype_bytes: int = 2,   # bfloat16 for params
    grad_dtype_bytes: int = 4,    # float32 for grads
    batch_size: int = 16,
    seq_len: int = 512,
    hidden_dim: int = 4096,
) -> MemoryEstimate:
    """Estimate per-GPU memory for a given model and sharding strategy.

    Optimizer state multipliers (per param, float32):
      SGD:      0  additional (no moment)
      SGD+mom:  1  additional (momentum buffer)
      Adam:     2  additional (m + v)
    """
    opt_state_multiplier = {"sgd": 0, "sgd_momentum": 1, "adam": 2, "adamw": 2}.get(
        optimizer.lower(), 2
    )

    shard_factor = {
        "FULL_SHARD": num_gpus,
        "SHARD_GRAD_OP": num_gpus,   # params replicated post-fwd but freed; grads/opt sharded
        "NO_SHARD": 1,
        "DDP": 1,
    }.get(sharding_strategy, 1)

    # Params
    params_gb = (param_count * param_dtype_bytes) / (shard_factor * 1e9)

    # Gradients (float32 for numerical stability)
    grad_shard = num_gpus if sharding_strategy in ("FULL_SHARD", "SHARD_GRAD_OP") else 1
    gradients_gb = (param_count * grad_dtype_bytes) / (grad_shard * 1e9)

    # Optimizer states (always float32 for Adam m + v)
    opt_shard = num_gpus if sharding_strategy in ("FULL_SHARD", "SHARD_GRAD_OP") else 1
    optimizer_states_gb = (param_count * 4 * opt_state_multiplier) / (opt_shard * 1e9)

    # Rough activation estimate (depends heavily on model arch; this is a proxy)
    activations_gb = (batch_size * seq_len * hidden_dim * 4) / 1e9  # float32 activations

    total_gb = params_gb + gradients_gb + optimizer_states_gb + activations_gb
    return MemoryEstimate(
        params_gb=round(params_gb, 2),
        gradients_gb=round(gradients_gb, 2),
        optimizer_states_gb=round(optimizer_states_gb, 2),
        activations_gb=round(activations_gb, 2),
        total_gb=round(total_gb, 2),
    )


# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------

configs = [
    ("7B",  7_000_000_000, "adam", "DDP",        8,  4096),
    ("7B",  7_000_000_000, "adam", "FULL_SHARD",  8,  4096),
    ("7B",  7_000_000_000, "adam", "FULL_SHARD", 16,  4096),
    ("13B", 13_000_000_000,"adam", "FULL_SHARD",  8,  5120),
    ("70B", 70_000_000_000,"adam", "FULL_SHARD", 64,  8192),
]

print(f"{'Model':<6} {'Strategy':<12} {'GPUs':>5} {'Params':>8} {'Grads':>7} {'Opt':>7} {'Act':>6} {'Total':>8} {'Fits 80GB?':>10}")
print("-" * 80)
for model_name, params, opt, strategy, n_gpus, hdim in configs:
    est = estimate_memory_per_gpu(params, opt, strategy, n_gpus, hidden_dim=hdim)
    fits = "YES" if est.total_gb < 80 else "NO"
    print(
        f"{model_name:<6} {strategy:<12} {n_gpus:>5} "
        f"{est.params_gb:>7.1f}G {est.gradients_gb:>6.1f}G "
        f"{est.optimizer_states_gb:>6.1f}G {est.activations_gb:>5.1f}G "
        f"{est.total_gb:>7.1f}G {fits:>10}"
    )

## Ray Data for Distributed Preprocessing

When the bottleneck is data processing (image decoding, resizing, captioning, tokenizing), not training, the fix is distributed CPU preprocessing — not more GPUs.

**Ray Data** is a distributed dataset library built on Ray Core:
- Scales across CPUs (and optionally GPUs) across a cluster
- Operations: `read → map → filter → flatmap → map_batches → write`
- Lazy execution with streaming — doesn't require the full dataset in memory
- Can use fractional GPU resources (e.g., `num_gpus=0.25` for 4 workers per GPU)

**Comparison with alternatives:**
| Tool | Strength | Weakness |
|---|---|---|
| `multiprocessing` | Simple, no deps | Single node only |
| Apache Spark | Mature, huge ecosystem | JVM overhead, less ML-native |
| Dask | Python-native, pandas-compatible | Less ML-native than Ray |
| Ray Data | ML-native, GPU support, streaming | Younger ecosystem |

**Production pattern:**
```
Ray cluster preprocessing
  → write formatted Parquet/WebDataset to cloud storage (S3/GCS)
  → training job reads pre-processed data (no CPU bottleneck during training)
```

This decouples preprocessing and training timelines, allows resumable pipelines, and lets you reuse processed data across multiple training runs.

In [ ]:
# ---------------------------------------------------------------------------
# Ray Data conceptual pipeline (requires ray[data] installed)
# ---------------------------------------------------------------------------

# import ray
#
# ray.init(address="auto")  # connect to running cluster, or ray.init() for local
#
# def resize_image(row: dict) -> dict:
#     """CPU-bound: decode + resize."""
#     img = decode_jpeg(row["bytes"])
#     row["image"] = resize(img, (224, 224))
#     return row
#
# def quality_filter(row: dict) -> bool:
#     return row["image"].shape == (224, 224, 3)
#
# def generate_caption(row: dict) -> dict:
#     """GPU-bound: run CLIP or similar."""
#     row["caption_embedding"] = clip_model.encode(row["image"])
#     return row
#
# ds = ray.data.read_images("s3://bucket/raw/")      # lazy read
# ds = ds.map(resize_image, num_cpus=1)               # distributed CPU map
# ds = ds.filter(quality_filter)                      # distributed filter
# ds = ds.map(generate_caption, num_gpus=0.5)         # fractional GPU per task
# ds = ds.write_parquet("s3://bucket/processed/")     # streaming write

print("Ray Data pipeline pattern shown above (commented).")


# ---------------------------------------------------------------------------
# Local simulation of the distributed pipeline pattern
# Uses multiprocessing.Pool as a stand-in for Ray workers
# ---------------------------------------------------------------------------

T = Any  # type alias for pipeline record


@dataclass
class DistributedPipeline:
    """Local simulation of a distributed map/filter/batch pipeline."""
    data: list[T]
    num_workers: int = 4

    def map(self, fn: Callable[[T], T]) -> "DistributedPipeline":
        """Apply fn to every element in parallel."""
        with multiprocessing.Pool(self.num_workers) as pool:
            result = pool.map(fn, self.data)
        return DistributedPipeline(result, self.num_workers)

    def filter(self, fn: Callable[[T], bool]) -> "DistributedPipeline":
        """Keep elements where fn returns True."""
        return DistributedPipeline([x for x in self.data if fn(x)], self.num_workers)

    def batch(self, size: int) -> list[list[T]]:
        """Yield batches of elements."""
        return [self.data[i : i + size] for i in range(0, len(self.data), size)]

    def __len__(self) -> int:
        return len(self.data)


# Synthetic workload: simulate image resize (sleep + arithmetic)
def synthetic_resize(item: dict) -> dict:
    """Simulate CPU-bound image processing."""
    # Simulate work via a tight loop (instead of sleep, to use actual CPU)
    arr = np.random.randn(224, 224).astype(np.float32)
    arr = arr / arr.max()  # normalize — forces actual CPU work
    return {"id": item["id"], "processed": True, "mean": float(arr.mean())}


def quality_filter(item: dict) -> bool:
    return item.get("processed", False)


dataset = [{"id": i, "processed": False} for i in range(200)]

worker_counts = [1, 2, 4]
times: list[float] = []

for n_workers in worker_counts:
    pipeline = DistributedPipeline(dataset, num_workers=n_workers)
    t0 = time.perf_counter()
    result = pipeline.map(synthetic_resize).filter(quality_filter)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f"workers={n_workers} | records={len(result)} | time={elapsed:.2f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar([str(w) for w in worker_counts], times, color=["#2196F3", "#4CAF50", "#FF9800"])
ax1.set_xlabel("Worker count")
ax1.set_ylabel("Wall time (s)")
ax1.set_title("Pipeline Wall Time vs Workers")

speedups = [times[0] / t for t in times]
ideal_speedup = worker_counts
ax2.plot(worker_counts, ideal_speedup, "k--", label="ideal", alpha=0.5)
ax2.plot(worker_counts, speedups, "b-o", label="measured")
ax2.set_xlabel("Worker count")
ax2.set_ylabel("Speedup")
ax2.set_title("Pipeline Speedup (vs 1 worker)")
ax2.legend()
ax2.set_xticks(worker_counts)

plt.tight_layout()
plt.show()

## Orchestration: Airflow, Prefect, K8s

Distributed training and data processing components need to be wired together into end-to-end workflows. That's the job of the orchestration layer.

**Prefect / Airflow — DAG-based task orchestration:**
- Define pipeline stages as `@task` functions, compose them into `@flow` (Prefect) or `DAG` (Airflow)
- Handles: dependency resolution, retries with backoff, scheduling (cron), monitoring, alerting
- Airflow: older, more enterprise-prevalent, Python-first DAGs, rich ecosystem
- Prefect: modern API, native async, easier local dev/test, better type ergonomics

**Kubernetes — container orchestration:**
- Schedule training `Job`s on GPU nodes: request `nvidia.com/gpu: 4`, K8s finds a node with capacity
- `tolerations` + `nodeSelector` route jobs to GPU node pools
- Handles node failure: restarts pods, respects `restartPolicy`
- `PersistentVolumeClaim` mounts shared storage (checkpoints, data)
- `KubeflowPipelines` (KFP) or `Argo Workflows` sit on top of K8s for ML-specific orchestration

**Typical production stack:**
```
Prefect flow triggers
  └─→ K8s Job: Ray preprocessing cluster
         └─→ Ray Data pipeline: read raw → process → write Parquet to S3
  └─→ K8s Job: distributed training (torchrun / FSDP)
         └─→ reads Parquet from S3
         └─→ writes checkpoints to artifact registry (MLflow / W&B)
  └─→ K8s Job: evaluation + model promotion
```

In [ ]:
# ---------------------------------------------------------------------------
# Prefect flow definition (conceptual)
# ---------------------------------------------------------------------------

# from prefect import flow, task
# from pathlib import Path
#
# @task(retries=3, retry_delay_seconds=60)
# def download_data(source: str) -> Path:
#     """Download raw data from source to local staging area."""
#     ...
#
# @task(retries=2)
# def preprocess(data_path: Path) -> Path:
#     """Run distributed preprocessing via Ray."""
#     ...
#
# @task
# def validate(processed_path: Path) -> bool:
#     """Run data quality checks; return False to abort pipeline."""
#     ...
#
# @task
# def upload(processed_path: Path, dest: str) -> str:
#     """Upload processed data to artifact store; return artifact URI."""
#     ...
#
# @flow(name="data-pipeline", log_prints=True)
# def data_pipeline(source: str, dest: str) -> str | None:
#     raw = download_data(source)
#     processed = preprocess(raw)
#     if validate(processed):
#         return upload(processed, dest)
#     print("Validation failed — aborting upload")
#     return None

print("Prefect flow definition shown above (commented).")


# ---------------------------------------------------------------------------
# Kubernetes Job YAML for a distributed training run
# ---------------------------------------------------------------------------

K8S_TRAINING_JOB_YAML = """
apiVersion: batch/v1
kind: Job
metadata:
  name: distributed-training-run-001
  namespace: ml-training
spec:
  completions: 4        # 4 pods total (one per GPU node)
  parallelism: 4        # all 4 run simultaneously
  backoffLimit: 2       # retry the job up to 2 times on failure
  template:
    spec:
      restartPolicy: OnFailure
      # Route to GPU nodes via nodeSelector
      nodeSelector:
        cloud.google.com/gke-accelerator: nvidia-tesla-a100
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      containers:
        - name: trainer
          image: my-registry/training:v1.2.3
          command:
            - torchrun
            - --nproc_per_node=4
            - --nnodes=4
            - --rdzv_backend=c10d
            - --rdzv_endpoint=$(MASTER_ADDR):29500
            - train.py
            - --config=config/7b_fsdp.yaml
          env:
            - name: MASTER_ADDR
              valueFrom:
                fieldRef:
                  fieldPath: status.hostIP
          resources:
            limits:
              nvidia.com/gpu: "4"   # 4 GPUs per pod
              memory: "256Gi"
              cpu: "32"
          volumeMounts:
            - name: data-volume
              mountPath: /data
            - name: checkpoint-volume
              mountPath: /checkpoints
      volumes:
        - name: data-volume
          persistentVolumeClaim:
            claimName: training-data-pvc
        - name: checkpoint-volume
          persistentVolumeClaim:
            claimName: checkpoint-pvc
"""

print("K8s Job YAML:")
print(K8S_TRAINING_JOB_YAML)


# ---------------------------------------------------------------------------
# Local DAG executor that simulates the orchestration pattern
# ---------------------------------------------------------------------------

@dataclass
class Task:
    name: str
    fn: Callable[..., Any]
    depends_on: list[str] = field(default_factory=list)


class LocalDAG:
    """Minimal DAG executor — topological sort + sequential execution."""

    def __init__(self) -> None:
        self._tasks: dict[str, Task] = {}

    def register(self, task: Task) -> None:
        self._tasks[task.name] = task

    def _topo_sort(self) -> list[str]:
        visited: set[str] = set()
        order: list[str] = []

        def dfs(name: str) -> None:
            if name in visited:
                return
            for dep in self._tasks[name].depends_on:
                dfs(dep)
            visited.add(name)
            order.append(name)

        for name in self._tasks:
            dfs(name)
        return order

    def run(self, context: dict[str, Any] | None = None) -> dict[str, Any]:
        """Execute all tasks in dependency order, passing results via context."""
        ctx: dict[str, Any] = context or {}
        for name in self._topo_sort():
            task = self._tasks[name]
            print(f"  [DAG] running task: {name}")
            t0 = time.perf_counter()
            result = task.fn(ctx)
            ctx[name] = result
            elapsed = time.perf_counter() - t0
            print(f"  [DAG] {name} done in {elapsed:.3f}s -> {result}")
        return ctx


# Simulated pipeline tasks
def download_data(ctx: dict) -> Path:
    time.sleep(0.05)  # simulate network I/O
    return Path("/tmp/raw_data")


def preprocess(ctx: dict) -> Path:
    raw = ctx["download_data"]
    time.sleep(0.1)   # simulate CPU processing
    return Path("/tmp/processed_data")


def validate(ctx: dict) -> bool:
    processed = ctx["preprocess"]
    time.sleep(0.02)
    return processed.parent.exists()  # trivial check in simulation


def upload(ctx: dict) -> str:
    if not ctx["validate"]:
        raise RuntimeError("Validation failed — refusing to upload")
    time.sleep(0.05)
    return "s3://bucket/processed/run_001"


dag = LocalDAG()
dag.register(Task("download_data", download_data))
dag.register(Task("preprocess",    preprocess,   depends_on=["download_data"]))
dag.register(Task("validate",      validate,     depends_on=["preprocess"]))
dag.register(Task("upload",        upload,       depends_on=["validate"]))

print("Running local DAG simulation:")
results = dag.run()
print(f"\nFinal artifact: {results['upload']}")

## Checkpoint

**Decision matrix:**

| Scenario | Solution |
|---|---|
| Model fits on 1 GPU, want faster training | DDP |
| Model doesn't fit on 1 GPU | FSDP |
| Data preprocessing is the bottleneck | Ray Data |
| Need to orchestrate multi-step pipelines | Prefect/Airflow |
| Need to manage GPU cluster | Kubernetes |
| All of the above at scale | All of the above, layered |

**Key takeaways:**

- **DDP**: near-linear scaling, minimal code changes, use when model fits on 1 GPU. Ring all-reduce is O(2N) bytes regardless of GPU count.
- **FSDP**: shards params + grads + optimizer states, enables training models that don't fit on 1 GPU. Trade-off: more communication (all-gather per layer) for less memory per GPU.
- **Ray**: distributed data processing, complementary to training frameworks. Decouples preprocessing from training — process once, train many times.
- **Orchestration**: the glue that holds production ML pipelines together. Handles retries, monitoring, dependencies, scheduling.
- In practice: you won't build DDP from scratch, but understanding the tradeoffs helps you design systems that use these components correctly.